In [1]:
import requests
import pandas as pd
import itertools

BASE_URL = "https://statistikdatabasen.scb.se/api/v2"
LANG = "sv"

TABLE_FOLKM_DESO = "TAB6574"
TABLE_INKOMST    = "TAB6679"
TABLE_HUSHALL    = "TAB6568"
TABLE_BOENDE     = "TAB6638"

BATCH_SIZE = 10

In [2]:
def get_metadata(table_id):
    url = f"{BASE_URL}/tables/{table_id}/metadata?lang={LANG}"
    r = requests.get(url)
    r.raise_for_status()
    return r.json()

def stockholm_deso_koder(meta):
    """Hämta alla Stockholm DeSO-koder med _DeSO2025 suffix"""
    region_cats = meta["dimension"]["Region"]["category"]["index"]
    koder = [k for k in region_cats.keys() 
             if k.startswith("0180") and k.endswith("_DeSO2025")]
    print(f"  {len(koder)} DeSO-koder hittade")
    return koder

def jsonstat2_to_df(data):
    dims = data["id"]
    values = data["value"]
    dim_vals = [list(data["dimension"][d]["category"]["label"].values()) for d in dims]
    df = pd.DataFrame(list(itertools.product(*dim_vals)), columns=dims)
    df["value"] = values
    return df

def fetch_batch(table_id, koder, extra_filters):
    """Hämta data i batcher om 10 regioner"""
    batches = [koder[i:i + BATCH_SIZE] for i in range(0, len(koder), BATCH_SIZE)]
    print(f"Hämtar {len(koder)} områden i {len(batches)} batcher...")
    
    dfs = []
    for i, batch in enumerate(batches, 1):
        # Bygg URL
        params = [
            f"lang={LANG}",
            "outputFormat=JSON-stat2",
            f"valueCodes[Region]={','.join(batch)}",
        ]
        for dim, vals in extra_filters.items():
            encoded_vals = [v.replace("+", "%2B") for v in vals]
            params.append(f"valueCodes[{dim}]={','.join(encoded_vals)}")
        
        url = f"{BASE_URL}/tables/{table_id}/data?" + "&".join(params)
        
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200:
                df = jsonstat2_to_df(r.json())
                dfs.append(df)
            else:
                print(f"  ⚠️  Batch {i}/{len(batches)} misslyckades – HTTP {r.status_code}")
        except Exception as e:
            print(f"  ⚠️  Batch {i}/{len(batches)} misslyckades – {str(e)[:50]}")
        
        if i % 10 == 0:
            print(f"  ✓ {i}/{len(batches)} batcher klara")
    
    if not dfs:
        print("❌ Ingen data hämtades.")
        return None
    
    result = pd.concat(dfs, ignore_index=True)
    print(f"✅ Klart – {len(result)} rader, {result['Region'].nunique()} områden")
    return result

print("Funktioner laddade")

Funktioner laddade


---
## 1. Folkmängd (TAB6574)
Total befolkning per DeSO-område

In [3]:
print("\n" + "="*60)
print("FOLKMÄNGD")
print("="*60)

meta_folkm = get_metadata(TABLE_FOLKM_DESO)
koder = stockholm_deso_koder(meta_folkm)

df_folkm = fetch_batch(
    TABLE_FOLKM_DESO,
    koder,
    {
        "Alder": ["totalt"],
        "Kon": ["1+2"],
        "ContentsCode": ["000007Y7"],
        "Tid": ["2024"],
    }
)

if df_folkm is not None:
    display(df_folkm.head(10))
    print(f"\n📊 Total befolkning: {df_folkm['value'].sum():,.0f}")
    print(f"   Snitt per område: {df_folkm['value'].mean():,.0f}")
    print(f"   Minsta område: {df_folkm['value'].min():,.0f}")
    print(f"   Största område: {df_folkm['value'].max():,.0f}")


FOLKMÄNGD
  569 DeSO-koder hittade
Hämtar 569 områden i 57 batcher...
  ✓ 10/57 batcher klara
  ✓ 20/57 batcher klara
  ✓ 30/57 batcher klara
  ✓ 40/57 batcher klara
  ✓ 50/57 batcher klara
✅ Klart – 569 rader, 569 områden


,Region,Alder,Kon,ContentsCode,Tid,value
0,0180C1010,totalt,totalt,Antal,2024,1809
1,0180C1020,totalt,totalt,Antal,2024,1881
2,0180C1030,totalt,totalt,Antal,2024,1928
3,0180C1040,totalt,totalt,Antal,2024,2685
4,0180C1050,totalt,totalt,Antal,2024,2535
5,0180C1060,totalt,totalt,Antal,2024,2058
6,0180C1070,totalt,totalt,Antal,2024,1324
7,0180C1080,totalt,totalt,Antal,2024,1542
8,0180C1090,totalt,totalt,Antal,2024,1577
9,0180C1101,totalt,totalt,Antal,2024,1936



📊 Total befolkning: 995,574
   Snitt per område: 1,750
   Minsta område: 827
   Största område: 3,064


---
## 2. Inkomst (TAB6679)
Sammanräknad förvärvsinkomst (medelvärde) per DeSO-område

In [4]:
print("\n" + "="*60)
print("INKOMST")
print("="*60)

meta_inkomst = get_metadata(TABLE_INKOMST)
koder = stockholm_deso_koder(meta_inkomst)

df_inkomst = fetch_batch(
    TABLE_INKOMST,
    koder,
    {
        "InkomstTyp": ["SaFörInk"],  # Sammanräknad förvärvsinkomst
        "Kon": ["1+2"],
        "ContentsCode": ["0000089T"],  # Medelvärde, tkr
        "Tid": ["2023"],
    }
)

if df_inkomst is not None:
    display(df_inkomst.head(10))
    print(f"\n💰 Medelinkomst: {df_inkomst['value'].mean():,.0f} tkr")
    print(f"   Median: {df_inkomst['value'].median():,.0f} tkr")
    print(f"   Lägsta: {df_inkomst['value'].min():,.0f} tkr")
    print(f"   Högsta: {df_inkomst['value'].max():,.0f} tkr")


INKOMST
  569 DeSO-koder hittade
Hämtar 569 områden i 57 batcher...
  ✓ 10/57 batcher klara
  ✓ 20/57 batcher klara
  ✓ 30/57 batcher klara
  ✓ 40/57 batcher klara
  ✓ 50/57 batcher klara
✅ Klart – 569 rader, 569 områden


,Region,InkomstTyp,Kon,ContentsCode,Tid,value
0,0180C1010,sammanräknad förvärvsinkomst,totalt,"Medelvärde, tkr",2023,None
1,0180C1020,sammanräknad förvärvsinkomst,totalt,"Medelvärde, tkr",2023,None
2,0180C1030,sammanräknad förvärvsinkomst,totalt,"Medelvärde, tkr",2023,None
3,0180C1040,sammanräknad förvärvsinkomst,totalt,"Medelvärde, tkr",2023,None
4,0180C1050,sammanräknad förvärvsinkomst,totalt,"Medelvärde, tkr",2023,None
5,0180C1060,sammanräknad förvärvsinkomst,totalt,"Medelvärde, tkr",2023,None
6,0180C1070,sammanräknad förvärvsinkomst,totalt,"Medelvärde, tkr",2023,None
7,0180C1080,sammanräknad förvärvsinkomst,totalt,"Medelvärde, tkr",2023,None
8,0180C1090,sammanräknad förvärvsinkomst,totalt,"Medelvärde, tkr",2023,None
9,0180C1101,sammanräknad förvärvsinkomst,totalt,"Medelvärde, tkr",2023,None



💰 Medelinkomst: nan tkr
   Median: nan tkr
   Lägsta: nan tkr
   Högsta: nan tkr


c:\Users\chris\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


---
## 3. Hushåll (TAB6568)
Totalt antal hushåll per DeSO-område

In [5]:
print("\n" + "="*60)
print("HUSHÅLL")
print("="*60)

meta_hushall = get_metadata(TABLE_HUSHALL)
koder = stockholm_deso_koder(meta_hushall)

df_hushall = fetch_batch(
    TABLE_HUSHALL,
    koder,
    {
        "Hushallstyp": ["TOTALT"],  # Totalt antal hushåll
        "ContentsCode": ["000007Y1"],  # Antal hushåll
        "Tid": ["2024"],
    }
)

if df_hushall is not None:
    display(df_hushall.head(10))
    print(f"\n🏠 Totalt hushåll: {df_hushall['value'].sum():,.0f}")
    print(f"   Snitt per område: {df_hushall['value'].mean():,.0f}")


HUSHÅLL
  569 DeSO-koder hittade
Hämtar 569 områden i 57 batcher...
  ✓ 10/57 batcher klara
  ✓ 20/57 batcher klara
  ✓ 30/57 batcher klara
  ✓ 40/57 batcher klara
  ✓ 50/57 batcher klara
✅ Klart – 569 rader, 569 områden


,Region,Hushallstyp,ContentsCode,Tid,value
0,0180C1010,totalt antal hushåll,Antal hushåll,2024,894
1,0180C1020,totalt antal hushåll,Antal hushåll,2024,931
2,0180C1030,totalt antal hushåll,Antal hushåll,2024,877
3,0180C1040,totalt antal hushåll,Antal hushåll,2024,1249
4,0180C1050,totalt antal hushåll,Antal hushåll,2024,1276
5,0180C1060,totalt antal hushåll,Antal hushåll,2024,1033
6,0180C1070,totalt antal hushåll,Antal hushåll,2024,665
7,0180C1080,totalt antal hushåll,Antal hushåll,2024,717
8,0180C1090,totalt antal hushåll,Antal hushåll,2024,673
9,0180C1101,totalt antal hushåll,Antal hushåll,2024,964



🏠 Totalt hushåll: 485,661
   Snitt per område: 854


---
## 4. Boende (TAB6638)
Antal bostäder per upplåtelseform och DeSO-område

In [6]:
print("\n" + "="*60)
print("BOENDE")
print("="*60)

meta_boende = get_metadata(TABLE_BOENDE)
koder = stockholm_deso_koder(meta_boende)

df_boende = fetch_batch(
    TABLE_BOENDE,
    koder,
    {
        "Upplatelseform": ["1", "2", "3", "ÖVRIGT"],
        "ContentsCode": ["00000864"],
        "Tid": ["2024"],
    }
)

if df_boende is not None:
    display(df_boende.head(10))
    print(f"\n🔑 Totalt bostäder: {df_boende['value'].sum():,.0f}")
    
    print(f"\n   Fördelning per upplåtelseform:")
    boende_sum = df_boende.groupby('Upplatelseform')['value'].sum()
    for form, antal in boende_sum.items():
        procent = (antal / boende_sum.sum()) * 100
        print(f"     {form}: {antal:,.0f} ({procent:.1f}%)")


BOENDE
  569 DeSO-koder hittade
Hämtar 569 områden i 57 batcher...
  ✓ 10/57 batcher klara
  ✓ 20/57 batcher klara
  ✓ 30/57 batcher klara
  ✓ 40/57 batcher klara
  ✓ 50/57 batcher klara
✅ Klart – 2276 rader, 569 områden


,Region,Upplatelseform,ContentsCode,Tid,value
0,0180C1010,hyresrätt,Antal,2024,974
1,0180C1010,bostadsrätt,Antal,2024,83
2,0180C1010,äganderätt,Antal,2024,30
3,0180C1010,uppgift saknas,Antal,2024,0
4,0180C1020,hyresrätt,Antal,2024,671
5,0180C1020,bostadsrätt,Antal,2024,237
6,0180C1020,äganderätt,Antal,2024,0
7,0180C1020,uppgift saknas,Antal,2024,0
8,0180C1030,hyresrätt,Antal,2024,168
9,0180C1030,bostadsrätt,Antal,2024,444



🔑 Totalt bostäder: 522,654

   Fördelning per upplåtelseform:
     bostadsrätt: 257,568 (49.3%)
     hyresrätt: 223,826 (42.8%)
     uppgift saknas: 1 (0.0%)
     äganderätt: 41,259 (7.9%)


---
## Sammanfattning och export

In [7]:
print("\n" + "="*60)
print("SAMMANFATTNING - STOCKHOLMS KOMMUN (569 DeSO-OMRÅDEN)")
print("="*60)

if df_folkm is not None:
    print(f"\n📊 Folkmängd 2024:")
    print(f"   {df_folkm['Region'].nunique()} områden")
    print(f"   {df_folkm['value'].sum():,.0f} invånare totalt")
    
if df_inkomst is not None:
    print(f"\n💰 Inkomst 2023 (sammanräknad förvärvsinkomst):")
    print(f"   {df_inkomst['Region'].nunique()} områden")
    print(f"   {df_inkomst['value'].mean():,.0f} tkr medelinkomst")
    
if df_hushall is not None:
    print(f"\n🏠 Hushåll 2024:")
    print(f"   {df_hushall['Region'].nunique()} områden")
    print(f"   {df_hushall['value'].sum():,.0f} hushåll totalt")
    
if df_boende is not None:
    print(f"\n🔑 Boende 2024:")
    print(f"   {df_boende['Region'].nunique()} områden")
    print(f"   {df_boende['value'].sum():,.0f} bostäder totalt")

print("\n" + "="*60)
print("EXPORT TILL CSV")
print("="*60)

if df_folkm is not None:
    df_folkm.to_csv('stockholm_folkm_deso_2024.csv', index=False)
    print("✓ stockholm_folkm_deso_2024.csv")

if df_inkomst is not None:
    df_inkomst.to_csv('stockholm_inkomst_deso_2023.csv', index=False)
    print("✓ stockholm_inkomst_deso_2023.csv")

if df_hushall is not None:
    df_hushall.to_csv('stockholm_hushall_deso_2024.csv', index=False)
    print("✓ stockholm_hushall_deso_2024.csv")

if df_boende is not None:
    df_boende.to_csv('stockholm_boende_deso_2024.csv', index=False)
    print("✓ stockholm_boende_deso_2024.csv")

print("\n✅ KLART! Alla tabeller hämtade och exporterade.")


SAMMANFATTNING - STOCKHOLMS KOMMUN (569 DeSO-OMRÅDEN)

📊 Folkmängd 2024:
   569 områden
   995,574 invånare totalt

💰 Inkomst 2023 (sammanräknad förvärvsinkomst):
   569 områden
   nan tkr medelinkomst

🏠 Hushåll 2024:
   569 områden
   485,661 hushåll totalt

🔑 Boende 2024:
   569 områden
   522,654 bostäder totalt

EXPORT TILL CSV
✓ stockholm_folkm_deso_2024.csv
✓ stockholm_inkomst_deso_2023.csv
✓ stockholm_hushall_deso_2024.csv
✓ stockholm_boende_deso_2024.csv

✅ KLART! Alla tabeller hämtade och exporterade.


In [9]:
import requests
import pandas as pd

BASE_URL = "https://statistikdatabasen.scb.se/api/v2"
TABLE_INKOMST = "TAB6679"

# Bygg URL
test_koder = ['0180C1010_DeSO2025', '0180C1020_DeSO2025', '0180C1030_DeSO2025']
params = [
    "lang=sv",
    "outputFormat=JSON-stat2",
    f"valueCodes[Region]={','.join(test_koder)}",
    "valueCodes[InkomstTyp]=SaFörInk",
    "valueCodes[Kon]=1%2B2",
    "valueCodes[ContentsCode]=0000089T",
    "valueCodes[Tid]=2023",
]
url = f"{BASE_URL}/tables/{TABLE_INKOMST}/data?" + "&".join(params)

print("URL:")
print(url)
print("\n" + "="*60)

r = requests.get(url)
print(f"Status: {r.status_code}")

if r.status_code == 200:
    data = r.json()
    print("\nDatastruktur:")
    print(f"  Dimensioner: {data.get('id', [])}")
    print(f"  Antal värden: {len(data.get('value', []))}")
    print(f"\nFörsta 10 värden:")
    print(data.get('value', [])[:10])
else:
    print(f"\nFel: {r.text}")

URL:
https://statistikdatabasen.scb.se/api/v2/tables/TAB6679/data?lang=sv&outputFormat=JSON-stat2&valueCodes[Region]=0180C1010_DeSO2025,0180C1020_DeSO2025,0180C1030_DeSO2025&valueCodes[InkomstTyp]=SaFörInk&valueCodes[Kon]=1%2B2&valueCodes[ContentsCode]=0000089T&valueCodes[Tid]=2023

Status: 200

Datastruktur:
  Dimensioner: ['Region', 'InkomstTyp', 'Kon', 'ContentsCode', 'Tid']
  Antal värden: 3

Första 10 värden:
[None, None, None]


In [10]:
import requests

BASE_URL = "https://statistikdatabasen.scb.se/api/v2"
TABLE_INKOMST = "TAB6679"
test_kod = '0180C1010_DeSO2025'

# Testa olika år och inkomsttyper
tests = [
    ("SaFörInk", "0000089T", "2023", "Sammanräknad förvärvsinkomst, Medelvärde, 2023"),
    ("SaFörInk", "0000089T", "2022", "Sammanräknad förvärvsinkomst, Medelvärde, 2022"),
    ("SaFörInk", "0000089T", "2021", "Sammanräknad förvärvsinkomst, Medelvärde, 2021"),
    ("SaFörInk", "0000089O", "2023", "Sammanräknad förvärvsinkomst, Antal personer, 2023"),
    ("NeInk", "0000089T", "2023", "Nettoinkomst, Medelvärde, 2023"),
]

print("Testa olika kombinationer:\n")
for inkomst_typ, contents, tid, beskrivning in tests:
    params = [
        "lang=sv",
        "outputFormat=JSON-stat2",
        f"valueCodes[Region]={test_kod}",
        f"valueCodes[InkomstTyp]={inkomst_typ}",
        "valueCodes[Kon]=1%2B2",
        f"valueCodes[ContentsCode]={contents}",
        f"valueCodes[Tid]={tid}",
    ]
    url = f"{BASE_URL}/tables/{TABLE_INKOMST}/data?" + "&".join(params)
    
    r = requests.get(url)
    if r.status_code == 200:
        data = r.json()
        value = data.get('value', [None])[0]
        status = f"✓ {value}" if value is not None else "✗ None"
        print(f"{status:15} {beskrivning}")
    else:
        print(f"✗ HTTP {r.status_code:3}  {beskrivning}")

Testa olika kombinationer:

✗ None          Sammanräknad förvärvsinkomst, Medelvärde, 2023
✗ None          Sammanräknad förvärvsinkomst, Medelvärde, 2022
✗ None          Sammanräknad förvärvsinkomst, Medelvärde, 2021
✗ None          Sammanräknad förvärvsinkomst, Antal personer, 2023
✗ None          Nettoinkomst, Medelvärde, 2023


In [11]:
import requests

BASE_URL = "https://statistikdatabasen.scb.se/api/v2"
TABLE_INKOMST = "TAB6679"

# Hämta metadata för att se vilka regiontyper som finns
r = requests.get(f"{BASE_URL}/tables/{TABLE_INKOMST}/metadata?lang=sv")
meta = r.json()

# Hitta Stockholm-regioner som INTE är DeSO
region_cats = meta["dimension"]["Region"]["category"]["index"]
region_labels = meta["dimension"]["Region"]["category"]["label"]

print("Stockholm-regioner i inkomsttabellen:\n")
sthlm_regioner = [(k, region_labels[k]) for k in region_cats.keys() 
                   if k.startswith("0180")]

# Visa olika typer
deso = [r for r in sthlm_regioner if "_DeSO2025" in r[0]]
regso = [r for r in sthlm_regioner if "_RegSO2025" in r[0]]
other = [r for r in sthlm_regioner if "_DeSO2025" not in r[0] and "_RegSO2025" not in r[0]]

print(f"DeSO-områden (_DeSO2025): {len(deso)}")
print(f"RegSO-områden (_RegSO2025): {len(regso)}")
print(f"Övriga (kommun/stadsdelar): {len(other)}")

if regso:
    print(f"\nExempel RegSO (första 5):")
    for k, v in regso[:5]:
        print(f"  {k}: {v}")
        
if other:
    print(f"\nExempel Övriga (första 10):")
    for k, v in other[:10]:
        print(f"  {k}: {v}")

Stockholm-regioner i inkomsttabellen:

DeSO-områden (_DeSO2025): 569
RegSO-områden (_RegSO2025): 127
Övriga (kommun/stadsdelar): 672

Exempel RegSO (första 5):
  0180R001_RegSO2025: Stockholm (Abrahamsberg)
  0180R002_RegSO2025: Stockholm (Akalla)
  0180R003_RegSO2025: Stockholm (Alvik)
  0180R004_RegSO2025: Stockholm (Aspudden)
  0180R005_RegSO2025: Stockholm (Bagarmossen)

Exempel Övriga (första 10):
  0180: 0180 Stockholm
  0180C1010: 0180C1010
  0180C1020: 0180C1020
  0180C1030: 0180C1030
  0180C1040: 0180C1040
  0180C1050: 0180C1050
  0180C1060: 0180C1060
  0180C1070: 0180C1070
  0180C1080: 0180C1080
  0180C1090: 0180C1090


In [12]:
import requests

BASE_URL = "https://statistikdatabasen.scb.se/api/v2"
TABLE_INKOMST = "TAB6679"

# Testa med RegSO istället för DeSO
test_regso = ['0180R001_RegSO2025', '0180R002_RegSO2025', '0180R003_RegSO2025']

params = [
    "lang=sv",
    "outputFormat=JSON-stat2",
    f"valueCodes[Region]={','.join(test_regso)}",
    "valueCodes[InkomstTyp]=SaFörInk",
    "valueCodes[Kon]=1%2B2",
    "valueCodes[ContentsCode]=0000089T",  # Medelvärde
    "valueCodes[Tid]=2023",
]
url = f"{BASE_URL}/tables/{TABLE_INKOMST}/data?" + "&".join(params)

print("Testar RegSO (stadsdelar) istället för DeSO:")
print(url)
print("\n" + "="*60)

r = requests.get(url)
print(f"Status: {r.status_code}")

if r.status_code == 200:
    data = r.json()
    values = data.get('value', [])
    print(f"\nAntal värden: {len(values)}")
    print(f"Värden: {values}")
    
    if any(v is not None for v in values):
        print("\n✅ SUCCESS! RegSO har inkomstdata!")
        print("\nStadsdelar:")
        for i, regso in enumerate(test_regso):
            region_name = regso.replace('_RegSO2025', '').replace('0180R', 'R')
            print(f"  {region_name}: {values[i]} tkr")
    else:
        print("\n❌ Även RegSO har None-värden")

Testar RegSO (stadsdelar) istället för DeSO:
https://statistikdatabasen.scb.se/api/v2/tables/TAB6679/data?lang=sv&outputFormat=JSON-stat2&valueCodes[Region]=0180R001_RegSO2025,0180R002_RegSO2025,0180R003_RegSO2025&valueCodes[InkomstTyp]=SaFörInk&valueCodes[Kon]=1%2B2&valueCodes[ContentsCode]=0000089T&valueCodes[Tid]=2023

Status: 200

Antal värden: 3
Värden: [None, None, None]

❌ Även RegSO har None-värden


In [13]:
import requests

BASE_URL = "https://statistikdatabasen.scb.se/api/v2"
TABLE_INKOMST = "TAB6679"

# Testa med hela Stockholm kommun
params = [
    "lang=sv",
    "outputFormat=JSON-stat2",
    "valueCodes[Region]=0180",  # Hela Stockholm kommun
    "valueCodes[InkomstTyp]=SaFörInk",
    "valueCodes[Kon]=1%2B2",
    "valueCodes[ContentsCode]=0000089T",
    "valueCodes[Tid]=2023",
]
url = f"{BASE_URL}/tables/{TABLE_INKOMST}/data?" + "&".join(params)

print("Testar HELA Stockholm kommun (0180):")
print(url)
print("\n" + "="*60)

r = requests.get(url)
print(f"Status: {r.status_code}")

if r.status_code == 200:
    data = r.json()
    value = data.get('value', [None])[0]
    print(f"Värde: {value}")
    
    if value is not None:
        print(f"\n✅ Hela kommunen har data: {value} tkr medelinkomst")
        print("\n💡 Slutsats: Inkomstdata finns INTE på DeSO- eller RegSO-nivå")
        print("   Du kan bara få inkomst aggregerat för hela kommunen")
    else:
        print("\n❌ WTF - inte ens hela kommunen har data!")
        print("    Denna tabell verkar inte ha inkomstdata alls för Stockholm 2023")

Testar HELA Stockholm kommun (0180):
https://statistikdatabasen.scb.se/api/v2/tables/TAB6679/data?lang=sv&outputFormat=JSON-stat2&valueCodes[Region]=0180&valueCodes[InkomstTyp]=SaFörInk&valueCodes[Kon]=1%2B2&valueCodes[ContentsCode]=0000089T&valueCodes[Tid]=2023

Status: 200
Värde: 480

✅ Hela kommunen har data: 480 tkr medelinkomst

💡 Slutsats: Inkomstdata finns INTE på DeSO- eller RegSO-nivå
   Du kan bara få inkomst aggregerat för hela kommunen


In [14]:
import requests

# Sök efter inkomsttabeller i SCB
BASE_URL = "https://statistikdatabasen.scb.se/api/v2"

# Lista tabeller i statistikdatabasen (om API stödjer det)
# Alternativt: kolla manuellt på https://www.statistikdatabasen.scb.se/

In [15]:
import requests
import pandas as pd

BASE_URL = "https://statistikdatabasen.scb.se/api/v2"
TABLE_FOLKM = "TAB6574"

# Hämta metadata
r = requests.get(f"{BASE_URL}/tables/{TABLE_FOLKM}/metadata?lang=sv")
meta = r.json()

# RegSO-koder MED stadsdelsnamn
region_labels = meta["dimension"]["Region"]["category"]["label"]
regso_stockholm = {k: v for k, v in region_labels.items() 
                   if k.startswith("0180") and "_RegSO2025" in k}

print(f"Hittade {len(regso_stockholm)} stadsdelar:")
for kod, namn in list(regso_stockholm.items())[:10]:
    # Ta bort "Stockholm (" och ")" för att få rent stadsdelsnamn
    stadsdel = namn.replace("Stockholm (", "").replace(")", "")
    print(f"  {kod:30} -> {stadsdel}")

Hittade 127 stadsdelar:
  0180R001_RegSO2025             -> Abrahamsberg
  0180R002_RegSO2025             -> Akalla
  0180R003_RegSO2025             -> Alvik
  0180R004_RegSO2025             -> Aspudden
  0180R005_RegSO2025             -> Bagarmossen
  0180R006_RegSO2025             -> Bandhagen
  0180R007_RegSO2025             -> Beckomberga
  0180R008_RegSO2025             -> Björkhagen
  0180R009_RegSO2025             -> Blackeberg
  0180R010_RegSO2025             -> Bredäng
